# Notebook 15: Oracle Union Recall

**Why this notebook exists:** before spending effort building a reranker or a fine-tuning pipeline, we want to know the hard *ceiling* achievable by simply combining candidates from methods we've already run. This costs nothing -- every method below already has its top-1000-per-query results cached on disk from earlier notebooks. No re-encoding, no training, no reranker.

**What "oracle union recall" means:** for a given set of retrieval channels (e.g. MiniLM + BM25 + BGE), take the union of their top-N candidates for each query, and check what fraction of the ground-truth relevant companies appear *anywhere* in that combined pool -- regardless of rank. This is an oracle number: no reranker or fine-tuning of the ranking stage can ever exceed it, since it assumes perfect reordering of the pool. It only tells us the best-case ceiling of a "combine these channels, then rerank" strategy.

**Channels available (all cached, no new compute):**
- MiniLM (exact search, all-fields) -- our best single-channel result, Recall@1000 = 0.741
- BM25 (lexical)
- BGE (all-fields, prefix-fixed) -- Recall@1000 = 0.727
- BGE (summary-only, prefix-fixed) -- Recall@1000 = 0.707
- OpenAI `text-embedding-3-large`
- Nomic
- BGE-M3 (`BAAI/bge-m3`) -- Recall@1000 = 0.704
- GTE-large (`thenlper/gte-large`) -- Recall@1000 = 0.768
- E5-Mistral-7B-Instruct (`intfloat/e5-mistral-7b-instruct`) -- Recall@1000 = 0.274
- Linq-Embed-Mistral (`Linq-AI-Research/Linq-Embed-Mistral`) -- Recall@1000 = 0.749
- SFR-Embedding-Mistral (`Salesforce/SFR-Embedding-Mistral`) -- Recall@1000 = 0.746

Excluded for now: BGE-Multilingual-Gemma2 (notebook 15, still encoding) and NV-Embed-v2 (notebook 19, abandoned).

**What this tells us:** if the union ceiling clears ~0.85, a reranker over the combined pool is likely sufficient and fine-tuning may not be necessary. If it plateaus well below 0.85, the residual gap is a genuine coverage problem that only a better base model or fine-tuning can close -- and this number tells us how big that residual really is before committing to a training pipeline.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RESULT_DIR = Path('result/15_oracle_union_recall')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ -- ready')

[Setup] Result folder : result/15_oracle_union_recall/ -- ready


## 1. Load ground truth and all cached channel results

In [2]:
print('[Load] Loading production ground truth...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_CUTOFF = 1000

def get_relevant(query_id, top_k=K_CUTOFF):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

# Note: BGE-Multilingual-Gemma2 (notebook 15) is excluded -- still encoding, no final results.csv yet. NV-Embed-v2 (notebook 19) is excluded -- abandoned due to a library incompatibility.
CHANNELS = {
    'MiniLM':          'result/03_baseline_minilm/minilm_results.csv',
    'BM25':            'result/01_baseline_bm25/bm25_results.csv',
    'BGE_allfields':   'result/14_bge_prefix_allfields/bge_results_prefix.csv',
    'BGE_summary':     'result/13_bge_prefix_summary/bge_results_prefix.csv',
    'OpenAI_large':    'result/04_baseline_openai_large/openai_large_results.csv',
    'Nomic':           'result/05_baseline_nomic/nomic_results.csv',
    'BGE_M3':          'result/16_baseline_bgem3/bgem3_results.csv',
    'GTE_large':       'result/17_baseline_gte_large/gte_large_results.csv',
    'E5_Mistral':      'result/18_baseline_e5_mistral/e5_mistral_results.csv',
    'Linq_Mistral':    'result/20_baseline_linq_mistral/20_baseline_linq_mistral_results.csv',
    'SFR_Mistral':     'result/21_baseline_sfr_mistral/21_baseline_sfr_mistral_results.csv',
}

channel_data = {}
for name, path in CHANNELS.items():
    df = pd.read_csv(path, usecols=['query_id', 'rank', 'domain'])
    df = df[df['rank'] <= K_CUTOFF]
    # query_id -> ordered list of domains (top-K_CUTOFF for that channel)
    channel_data[name] = {
        qid: sub.sort_values('rank')['domain'].tolist()
        for qid, sub in df.groupby('query_id')
    }
    print(f'[Load] {name:<14} : {len(channel_data[name])} queries, top-{K_CUTOFF} cached')

[Load] Loading production ground truth...
[Load] MiniLM         : 101 queries, top-1000 cached
[Load] BM25           : 101 queries, top-1000 cached
[Load] BGE_allfields  : 101 queries, top-1000 cached
[Load] BGE_summary    : 101 queries, top-1000 cached
[Load] OpenAI_large   : 101 queries, top-1000 cached
[Load] Nomic          : 101 queries, top-1000 cached
[Load] BGE_M3         : 101 queries, top-1000 cached
[Load] GTE_large      : 101 queries, top-1000 cached
[Load] E5_Mistral     : 101 queries, top-1000 cached
[Load] Linq_Mistral   : 101 queries, top-1000 cached
[Load] SFR_Mistral    : 101 queries, top-1000 cached


## 2. Per-channel solo recall (sanity check against known numbers)

Confirms we're reading the same cached data that produced the numbers we already know, before computing anything new.

In [3]:
query_ids = sorted(channel_data['MiniLM'].keys())

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

print('[Sanity] Solo Recall@1000 per channel (should match prior notebooks):')
for name in CHANNELS:
    recalls = []
    for qid in query_ids:
        relevant = get_relevant(qid)
        retrieved = channel_data[name].get(qid, [])
        recalls.append(recall_at_k(retrieved, relevant, K_CUTOFF))
    print(f'  {name:<14} : {np.mean(recalls):.3f}')

[Sanity] Solo Recall@1000 per channel (should match prior notebooks):
  MiniLM         : 0.741
  BM25           : 0.635
  BGE_allfields  : 0.727
  BGE_summary    : 0.707
  OpenAI_large   : 0.728
  Nomic          : 0.684
  BGE_M3         : 0.704
  GTE_large      : 0.768
  E5_Mistral     : 0.274
  Linq_Mistral   : 0.749
  SFR_Mistral    : 0.746


## 3. Oracle union recall -- all pairwise combinations + full union

For every combination of channels, take the union of their cached top-1000 domains per query and compute recall against the same top-1000 ground truth. This is the oracle ceiling for "combine these channels + a perfect reranker".

In [4]:
from itertools import combinations

def oracle_union_recall(channel_names, k_per_channel=K_CUTOFF):
    recalls = []
    pool_sizes = []
    for qid in query_ids:
        relevant = get_relevant(qid)
        union_pool = set()
        for name in channel_names:
            union_pool.update(channel_data[name].get(qid, [])[:k_per_channel])
        pool_sizes.append(len(union_pool))
        recalls.append(len(union_pool & relevant) / len(relevant) if relevant else 0)
    return np.mean(recalls), np.mean(pool_sizes)

results_rows = []

# Individual channels (baseline reference)
for name in CHANNELS:
    r, pool = oracle_union_recall([name])
    results_rows.append({'combination': name, 'n_channels': 1, 'oracle_recall_at_1000': r, 'avg_pool_size': pool})

# All pairwise combinations
for pair in combinations(CHANNELS.keys(), 2):
    r, pool = oracle_union_recall(list(pair))
    results_rows.append({'combination': ' + '.join(pair), 'n_channels': 2, 'oracle_recall_at_1000': r, 'avg_pool_size': pool})

# MiniLM + each other channel individually (most likely practical pairing, since MiniLM is our best solo channel)
# (already covered by pairwise loop above -- included for completeness)

# Full union of all channels
all_names = list(CHANNELS.keys())
r_all, pool_all = oracle_union_recall(all_names)
results_rows.append({'combination': ' + '.join(all_names), 'n_channels': len(all_names), 'oracle_recall_at_1000': r_all, 'avg_pool_size': pool_all})

results_df = pd.DataFrame(results_rows).sort_values('oracle_recall_at_1000', ascending=False)
results_df.to_csv(RESULT_DIR / 'oracle_union_recall.csv', index=False)
print(results_df.to_string(index=False))
print(f'\n[Saved] {RESULT_DIR}/oracle_union_recall.csv')

                                                                                                                      combination  n_channels  oracle_recall_at_1000  avg_pool_size
MiniLM + BM25 + BGE_allfields + BGE_summary + OpenAI_large + Nomic + BGE_M3 + GTE_large + E5_Mistral + Linq_Mistral + SFR_Mistral          11               0.961446    2768.148515
                                                                                                            MiniLM + Linq_Mistral           2               0.869198    1300.029703
                                                                                                               MiniLM + GTE_large           2               0.862515    1262.297030
                                                                                                             MiniLM + SFR_Mistral           2               0.861901    1288.089109
                                                                                                    

## 4. Pool-size sweep for the best pairing -- where does the ceiling stop improving?

For the best-performing channel pair, sweep how many candidates we take per channel (500 / 1000) to see if a smaller per-channel pool already captures most of the achievable ceiling (relevant for sizing a real reranker pool later).

In [5]:
best_pair_row = results_df[results_df['n_channels'] == 2].iloc[0]
best_pair = best_pair_row['combination'].split(' + ')
print(f'[Sweep] Best pair: {best_pair} (oracle recall@1000 = {best_pair_row["oracle_recall_at_1000"]:.3f})')

print()
print(f'  {"k_per_channel":<15} {"oracle_recall":>14} {"avg_pool_size":>14}')
for k_per_channel in [100, 250, 500, 750, 1000]:
    r, pool = oracle_union_recall(best_pair, k_per_channel=k_per_channel)
    print(f'  {k_per_channel:<15} {r:>14.3f} {pool:>14.1f}')

[Sweep] Best pair: ['MiniLM', 'Linq_Mistral'] (oracle recall@1000 = 0.869)

  k_per_channel    oracle_recall  avg_pool_size
  100                      0.158          172.2
  250                      0.351          396.5
  500                      0.599          724.1
  750                      0.770         1013.9
  1000                     0.869         1300.0
